In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.21.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
from google.colab import files
import os
import zipfile

uploaded = files.upload()

zip_path = next(
    (name for name in uploaded if name.lower().endswith(".zip")),
    None
)

if zip_path is None:
    raise ValueError("Please upload the Kaggle dataset ZIP file.")

print("Uploaded:", zip_path)
print("Size:", os.path.getsize(zip_path), "bytes")
print("Valid ZIP:", zipfile.is_zipfile(zip_path))

if not zipfile.is_zipfile(zip_path):
    raise ValueError("This file is not a valid ZIP archive.")

EXTRACT_PATH = "/content/skinsakhi_dataset"
os.makedirs(EXTRACT_PATH, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(EXTRACT_PATH)

TRAIN_DIR = None
TEST_DIR = None

for root, dirs, _ in os.walk(EXTRACT_PATH):
    if "train_set" in dirs and "test_set" in dirs:
        TRAIN_DIR = os.path.join(root, "train_set")
        TEST_DIR = os.path.join(root, "test_set")
        break

if TRAIN_DIR is None:
    raise FileNotFoundError(
        "Could not find train_set and test_set in the extracted dataset."
    )

print("\nDATASET READY!")
print("Training folder:", TRAIN_DIR)
print("Testing folder:", TEST_DIR)
print("Classes:", os.listdir(TRAIN_DIR))

Saving archive (1).zip to archive (1).zip
Uploaded: archive (1).zip
Size: 18126678 bytes
Valid ZIP: True

DATASET READY!
Training folder: /content/skinsakhi_dataset/skin-disease-datasaet/train_set
Testing folder: /content/skinsakhi_dataset/skin-disease-datasaet/test_set
Classes: ['VI-chickenpox', 'BA-impetigo', 'FU-nail-fungus', 'VI-shingles', 'BA- cellulitis', 'FU-athlete-foot', 'PA-cutaneous-larva-migrans', 'FU-ringworm']


In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/SkinSakhi"
os.makedirs(SAVE_DIR, exist_ok=True)

print("Google Drive connected!")
print("Model save folder:", SAVE_DIR)

Mounted at /content/drive
Google Drive connected!
Model save folder: /content/drive/MyDrive/SkinSakhi


In [ ]:

import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

print("GPU:", tf.config.list_physical_devices("GPU"))

class_names = sorted([
    name for name in os.listdir(TRAIN_DIR)
    if os.path.isdir(os.path.join(TRAIN_DIR, name))
])

train_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    class_names=class_names,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    class_names=class_names,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_ds = keras.utils.image_dataset_from_directory(
    TEST_DIR,
    class_names=class_names,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

base_model = keras.applications.MobileNetV3Small(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

inputs = keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(
    len(class_names),
    activation="softmax"
)(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

checkpoint_path = os.path.join(SAVE_DIR, "skinsakhi_best.keras")

checkpoint = keras.callbacks.ModelCheckpoint(
    checkpoint_path,
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[checkpoint]
)

model = keras.models.load_model(checkpoint_path)

test_loss, test_accuracy = model.evaluate(test_ds)
print(f"Test accuracy: {test_accuracy * 100:.2f}%")

model.save(os.path.join(SAVE_DIR, "skinsakhi_mobilenetv3.keras"))

with open(os.path.join(SAVE_DIR, "skinsakhi_classes.txt"), "w") as f:
    f.write("\n".join(class_names))

print("SUCCESS! Model and labels saved to Google Drive/SkinSakhi.")


GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Found 924 files belonging to 8 classes.
Using 740 files for training.
Found 924 files belonging to 8 classes.
Using 184 files for validation.
Found 233 files belonging to 8 classes.
4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 326ms/step - accuracy: 0.1525 - loss: 2.3975
Epoch 1: val_accuracy improved from None to 0.49457, saving model to /content/drive/MyDrive/SkinSakhi/skinsakhi_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/SkinSakhi/skinsakhi_best.keras
24/24 ━━━━━━━━━━━━━━━━━━━━ 35s 792ms/step - accuracy: 0.2284 - loss: 2.1318 - val_accuracy: 0.4946 - val_loss: 1.5439
Epoch 2/5
21/24 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.4810 - loss: 1.4837
Epoch 2: val_accuracy improved from 0.49457 to 0.73913, saving model to /content/drive/MyDrive/SkinSakhi/skinsakhi_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/SkinSakhi/skin

In [ ]:

import os
from tensorflow import keras

MODEL_PATH = "/content/drive/MyDrive/SkinSakhi/skinsakhi_mobilenetv3.keras"
LABEL_PATH = "/content/drive/MyDrive/SkinSakhi/skinsakhi_classes.txt"

print("Model exists:", os.path.exists(MODEL_PATH))
print("Labels exist:", os.path.exists(LABEL_PATH))

model = keras.models.load_model(MODEL_PATH)

with open(LABEL_PATH, "r") as f:
    class_names = f.read().splitlines()

print("Model loaded successfully!")
print("Number of classes:", len(class_names))
print("Classes:", class_names)


Model exists: True
Labels exist: True
Model loaded successfully!
Number of classes: 8
Classes: ['BA- cellulitis', 'BA-impetigo', 'FU-athlete-foot', 'FU-nail-fungus', 'FU-ringworm', 'PA-cutaneous-larva-migrans', 'VI-chickenpox', 'VI-shingles']


In [ ]:
import os
import numpy as np
from tensorflow import keras

folder = "/content/drive/MyDrive/SkinSakhi"

print("Model files:")
for name in os.listdir(folder):
    print(name)

model = keras.models.load_model(
    folder + "/skinsakhi_mobilenetv3.keras"
)

with open(folder + "/skinsakhi_classes.txt") as f:
    class_names = f.read().splitlines()

print("\nNumber of classes:", len(class_names))
print("Classes:", class_names)
print("Model loaded successfully!")

Model files:
skinsakhi_best.keras
skinsakhi_mobilenetv3.keras
skinsakhi_classes.txt

Number of classes: 8
Classes: ['BA- cellulitis', 'BA-impetigo', 'FU-athlete-foot', 'FU-nail-fungus', 'FU-ringworm', 'PA-cutaneous-larva-migrans', 'VI-chickenpox', 'VI-shingles']
Model loaded successfully!


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

test_dir = "/content/skinsakhi_dataset/skin-disease-dataset/test_set"

test_ds = keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(224, 224),
    batch_size=32,
    shuffle=False
)

y_true = np.concatenate([
    labels.numpy() for _, labels in test_ds
])

predictions = model.predict(test_ds, verbose=1)
y_pred = np.argmax(predictions, axis=1)

print("\nCLASS-WISE PERFORMANCE:\n")
print(classification_report(
    y_true,
    y_pred,
    labels=range(len(test_ds.class_names)),
    target_names=test_ds.class_names,
    zero_division=0
))

print("\nCONFUSION MATRIX:\n")
print(confusion_matrix(y_true, y_pred))

FileNotFoundError: [Errno 2] No such file or directory: '/content/skinsakhi_dataset/skin-disease-dataset/test_set'

In [ ]:
import os

for root, dirs, files in os.walk("/content/skinsakhi_dataset"):
    level = root.replace("/content/skinsakhi_dataset", "").count(os.sep)
    if level <= 3:
        print(root)
        for d in dirs:
            print("  Folder:", d)

/content/skinsakhi_dataset
  Folder: skin-disease-datasaet
/content/skinsakhi_dataset/skin-disease-datasaet
  Folder: test_set
  Folder: train_set
/content/skinsakhi_dataset/skin-disease-datasaet/test_set
  Folder: VI-chickenpox
  Folder: BA-impetigo
  Folder: FU-nail-fungus
  Folder: VI-shingles
  Folder: BA- cellulitis
  Folder: FU-athlete-foot
  Folder: PA-cutaneous-larva-migrans
  Folder: FU-ringworm
/content/skinsakhi_dataset/skin-disease-datasaet/test_set/VI-chickenpox
/content/skinsakhi_dataset/skin-disease-datasaet/test_set/BA-impetigo
/content/skinsakhi_dataset/skin-disease-datasaet/test_set/FU-nail-fungus
/content/skinsakhi_dataset/skin-disease-datasaet/test_set/VI-shingles
/content/skinsakhi_dataset/skin-disease-datasaet/test_set/BA- cellulitis
/content/skinsakhi_dataset/skin-disease-datasaet/test_set/FU-athlete-foot
/content/skinsakhi_dataset/skin-disease-datasaet/test_set/PA-cutaneous-larva-migrans
/content/skinsakhi_dataset/skin-disease-datasaet/test_set/FU-ringworm
/cont

In [ ]:
import os
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow import keras

base = "/content/skinsakhi_dataset"

test_dir = None

for root, dirs, files in os.walk(base):
    if os.path.basename(root) == "test_set":
        test_dir = root
        break

print("Test folder:", test_dir)

if test_dir is None:
    print("Test folder not found. Check the dataset extraction.")
else:
    test_ds = keras.utils.image_dataset_from_directory(
        test_dir,
        image_size=(224, 224),
        batch_size=32,
        shuffle=False
    )

    y_true = np.concatenate([y.numpy() for _, y in test_ds])
    probs = model.predict(test_ds, verbose=1)
    y_pred = np.argmax(probs, axis=1)

    print("\nCLASS-WISE RESULTS:\n")
    print(classification_report(
        y_true,
        y_pred,
        target_names=test_ds.class_names,
        zero_division=0
    ))

    print("\nCONFUSION MATRIX:\n")
    print(confusion_matrix(y_true, y_pred))



Test folder: /content/skinsakhi_dataset/skin-disease-datasaet/test_set
Found 233 files belonging to 8 classes.
8/8 ━━━━━━━━━━━━━━━━━━━━ 10s 683ms/step

CLASS-WISE RESULTS:

                            precision    recall  f1-score   support

            BA- cellulitis       0.82      0.94      0.87        33
               BA-impetigo       0.95      0.90      0.92        20
           FU-athlete-foot       0.91      0.97      0.94        32
            FU-nail-fungus       1.00      0.97      0.98        33
               FU-ringworm       0.94      0.65      0.77        23
PA-cutaneous-larva-migrans       0.65      0.60      0.62        25
             VI-chickenpox       1.00      0.94      0.97        34
               VI-shingles       0.79      0.94      0.86        33

                  accuracy                           0.88       233
                 macro avg       0.88      0.86      0.87       233
              weighted avg       0.88      0.88      0.88       233


CONFUSI

In [ ]:
import os

train_dir = "/content/skinsakhi_dataset/skin-disease-dataset/train_set"

for name in sorted(os.listdir(train_dir)):
    path = os.path.join(train_dir, name)
    if os.path.isdir(path):
        count = sum(
            1 for f in os.listdir(path)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
        )
        print(f"{name}: {count} images")


FileNotFoundError: [Errno 2] No such file or directory: '/content/skinsakhi_dataset/skin-disease-dataset/train_set'

In [ ]:
import os

base = "/content/skinsakhi_dataset"

for root, dirs, files in os.walk(base):
    if os.path.basename(root) == "train_set":
        train_dir = root
        break
else:
    raise FileNotFoundError("Training folder not found")

print("Correct path:", train_dir)
print("\nTraining images per class:")

for name in sorted(os.listdir(train_dir)):
    path = os.path.join(train_dir, name)
    if os.path.isdir(path):
        count = sum(
            f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
            for f in os.listdir(path)
        )
        print(f"{name}: {count}")

Correct path: /content/skinsakhi_dataset/skin-disease-datasaet/train_set

Training images per class:
BA- cellulitis: 136
BA-impetigo: 80
FU-athlete-foot: 124
FU-nail-fungus: 129
FU-ringworm: 90
PA-cutaneous-larva-migrans: 100
VI-chickenpox: 136
VI-shingles: 130


In [ ]:
best_model = keras.models.load_model(
    "/content/drive/MyDrive/SkinSakhi/skinsakhi_best.keras"
)

best_loss, best_acc = best_model.evaluate(test_ds, verbose=0)
print(f"Best model test accuracy: {best_acc * 100:.2f}%")

Best model test accuracy: 87.98%


In [ ]:
print("Current model:", model.name)
print("Best model:", best_model.name)
print("Training dataset:", train_dir)
print("Classes:", class_names)

Current model: functional
Best model: functional
Training dataset: /content/skinsakhi_dataset/skin-disease-datasaet/train_set
Classes: ['BA- cellulitis', 'BA-impetigo', 'FU-athlete-foot', 'FU-nail-fungus', 'FU-ringworm', 'PA-cutaneous-larva-migrans', 'VI-chickenpox', 'VI-shingles']


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import os

train_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(224, 224),
    batch_size=16,
    shuffle=True,
    seed=42
)

test_dir = "/content/skinsakhi_dataset/skin-disease-dataset/test_set"

test_ds = keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(224, 224),
    batch_size=16,
    shuffle=False
)

# Augmentation for training only
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1)
])

# Keep original model safe; fine-tune a copy
improved = keras.models.load_model(
    "/content/drive/MyDrive/SkinSakhi/skinsakhi_mobilenetv3.keras"
)

# Freeze feature layers and train the classification head
for layer in improved.layers:
    layer.trainable = False

improved.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Train with augmentation
aug_train_ds = train_ds.map(
    lambda x, y: (augment(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

improved.fit(aug_train_ds, validation_data=test_ds, epochs=3)

save_path = "/content/drive/MyDrive/SkinSakhi/skinsakhi_experimental.keras"
improved.save(save_path)

print("Saved separate experimental model:", save_path)

Found 924 files belonging to 8 classes.


FileNotFoundError: [Errno 2] No such file or directory: '/content/skinsakhi_dataset/skin-disease-dataset/test_set'

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

base = "/content/skinsakhi_dataset"
save_path = "/content/drive/MyDrive/SkinSakhi/skinsakhi_experimental.keras"

train_dir = None
test_dir = None

for root, dirs, files in os.walk(base):
    if os.path.basename(root) == "train_set":
        train_dir = root
    elif os.path.basename(root) == "test_set":
        test_dir = root

if train_dir is None or test_dir is None:
    raise FileNotFoundError(
        "Dataset folders not found. Check whether the dataset is extracted."
    )

print("Training folder:", train_dir)
print("Testing folder:", test_dir)

train_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(224, 224),
    batch_size=16,
    shuffle=True,
    seed=42
)

test_ds = keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(224, 224),
    batch_size=16,
    shuffle=False
)

print("Classes:", train_ds.class_names)

augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1)
])

model_path = "/content/drive/MyDrive/SkinSakhi/skinsakhi_mobilenetv3.keras"
improved = keras.models.load_model(model_path)

# Keep all original model layers frozen for this experiment
for layer in improved.layers:
    layer.trainable = False

improved.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

aug_train_ds = train_ds.map(
    lambda x, y: (augment(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

history = improved.fit(
    aug_train_ds,
    validation_data=test_ds,
    epochs=3
)

test_loss, test_acc = improved.evaluate(test_ds, verbose=0)
print(f"\nExperimental model test accuracy: {test_acc * 100:.2f}%")

improved.save(save_path)
print("Experimental model saved:", save_path)
print("Original model remains unchanged.")

Training folder: /content/skinsakhi_dataset/skin-disease-datasaet/train_set
Testing folder: /content/skinsakhi_dataset/skin-disease-datasaet/test_set
Found 924 files belonging to 8 classes.
Found 233 files belonging to 8 classes.
Classes: ['BA- cellulitis', 'BA-impetigo', 'FU-athlete-foot', 'FU-nail-fungus', 'FU-ringworm', 'PA-cutaneous-larva-migrans', 'VI-chickenpox', 'VI-shingles']
Epoch 1/3


/usr/local/lib/python3.13/dist-packages/keras/src/backend/tensorflow/trainer.py:86: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


58/58 ━━━━━━━━━━━━━━━━━━━━ 33s 402ms/step - accuracy: 0.7348 - loss: 0.8476 - val_accuracy: 0.8798 - val_loss: 0.6263
Epoch 2/3
58/58 ━━━━━━━━━━━━━━━━━━━━ 8s 138ms/step - accuracy: 0.7695 - loss: 0.8214 - val_accuracy: 0.8798 - val_loss: 0.6263
Epoch 3/3
58/58 ━━━━━━━━━━━━━━━━━━━━ 10s 139ms/step - accuracy: 0.7489 - loss: 0.8319 - val_accuracy: 0.8798 - val_loss: 0.6263

Experimental model test accuracy: 87.98%
Experimental model saved: /content/drive/MyDrive/SkinSakhi/skinsakhi_experimental.keras
Original model remains unchanged.


In [9]:

import os
import cv2
import numpy as np
import gradio as gr
from PIL import Image
from tensorflow import keras

BASE_PATH = "/content/drive/MyDrive/SkinSakhi"
MODEL_PATH = os.path.join(BASE_PATH, "skinsakhi_mobilenetv3.keras")
LABEL_PATH = os.path.join(BASE_PATH, "skinsakhi_classes.txt")

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

if not os.path.exists(LABEL_PATH):
    raise FileNotFoundError(f"Labels not found: {LABEL_PATH}")

model = keras.models.load_model(MODEL_PATH)

with open(LABEL_PATH, "r") as f:
    class_names = [line.strip() for line in f if line.strip()]

if len(class_names) != model.output_shape[-1]:
    raise ValueError("Class labels do not match model outputs.")


def is_document_or_paper(image):
    image = image.convert("RGB").resize((224, 224))
    rgb = np.asarray(image, dtype=np.uint8)

    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

    brightness = gray > 215
    low_saturation = hsv[:, :, 1] < 55

    bright_paper = brightness & low_saturation
    paper_ratio = float(np.mean(bright_paper))

    edges = cv2.Canny(gray, 60, 150)
    edge_ratio = float(np.mean(edges > 0))

    # Reject images dominated by a bright, low-colour paper background
    # with visible text or line details.
    if paper_ratio > 0.45 and edge_ratio > 0.025:
        return True

    # Reject nearly blank white or grey document images.
    if paper_ratio > 0.80 and float(np.std(gray)) < 35:
        return True

    return False


def screen_skin(image):
    if image is None:
        return "Please upload an image."

    image = image.convert("RGB")

    if image.width < 100 or image.height < 100:
        return "Image too small. Please upload a clearer image."

    if is_document_or_paper(image):
        return (
            "IMAGE NOT RELATED TO SKIN\n\n"
            "This image appears to contain paper, notes, or a document.\n\n"
            "Please upload a clear photo of a skin condition."
        )

    resized = image.resize((224, 224))
    arr = np.asarray(resized, dtype=np.float32)[None, ...]

    predictions = model.predict(arr, verbose=0)[0]

    if not np.all(np.isfinite(predictions)):
        return "Unable to assess this image. Please try another."

    top_indices = np.argsort(predictions)[::-1]
    top_index = int(top_indices[0])

    confidence = float(predictions[top_index])
    second_score = (
        float(predictions[top_indices[1]])
        if len(top_indices) > 1 else 0.0
    )

    if confidence < 0.50 or confidence - second_score < 0.10:
        return (
            "UNABLE TO ASSESS RELIABLY\n\n"
            "The model is uncertain about this image.\n"
            "Please upload a clearer skin photo or consult "
            "a healthcare professional."
        )

    return (
        "EXPERIMENTAL MODEL SUGGESTION\n\n"
        f"Suggested class: {class_names[top_index]}\n\n"
        "This is not a diagnosis. The model may be wrong, "
        "even when its score is high. Please consult a qualified "
        "healthcare professional for medical advice."
    )


with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="teal",
        secondary_hue="blue"
    ),
    title="SkinSakhi | Skin Screening Assistant"
) as demo:

    gr.Markdown("""
    # 🩺 SkinSakhi
    ### AI-Assisted Skin Condition Screening

    Upload a clear skin photo to receive an experimental model suggestion.
    """)

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(
                type="pil",
                label="Upload Skin Image"
            )

            analyze_btn = gr.Button(
                "Analyze Image",
                variant="primary"
            )

            clear_btn = gr.ClearButton([image_input])

        with gr.Column():
            result_output = gr.Textbox(
                label="Screening Result",
                lines=9,
                interactive=False
            )

    analyze_btn.click(
        fn=screen_skin,
        inputs=image_input,
        outputs=result_output
    )

    gr.Markdown("""
    ---
    **Medical disclaimer:** SkinSakhi is an experimental educational
    prototype, not a medical device. It cannot diagnose or rule out
    disease. The image-type check is heuristic and may make mistakes.
    Seek professional medical advice.
    """)

demo.launch(share=True)


/tmp/ipykernel_1504/18691712.py:105: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8a7ddb7a61b5086121.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import os
from tensorflow import keras

SAVE_DIR = "/content/drive/MyDrive/SkinSakhi"

MODEL_PATH = os.path.join(
    SAVE_DIR, "skinsakhi_mobilenetv3.keras"
)

LABEL_PATH = os.path.join(
    SAVE_DIR, "skinsakhi_classes.txt"
)

os.makedirs(SAVE_DIR, exist_ok=True)

print("Model exists:", os.path.exists(MODEL_PATH))
print("Labels exist:", os.path.exists(LABEL_PATH))

if os.path.exists(MODEL_PATH) and os.path.exists(LABEL_PATH):
    print("Model size:", round(os.path.getsize(MODEL_PATH) / (1024 * 1024), 2), "MB")

    with open(LABEL_PATH, "r") as f:
        labels = [line.strip() for line in f if line.strip()]

    print("Number of classes:", len(labels))
    print("Classes:", labels)

    model_check = keras.models.load_model(MODEL_PATH)
    print("Model output classes:", model_check.output_shape[-1])

    assert len(labels) == model_check.output_shape[-1], (
        "ERROR: Labels and model outputs do not match!"
    )

    print("CONFIRMATION: Saved model and labels match.")
else:
    print("Files missing. Check your Google Drive connection.")

Model exists: True
Labels exist: True
Model size: 4.17 MB
Number of classes: 8
Classes: ['BA- cellulitis', 'BA-impetigo', 'FU-athlete-foot', 'FU-nail-fungus', 'FU-ringworm', 'PA-cutaneous-larva-migrans', 'VI-chickenpox', 'VI-shingles']
Model output classes: 8
CONFIRMATION: Saved model and labels match.


In [ ]:

import os
from tensorflow import keras

folder = "/content/drive/MyDrive/SkinSakhi"

files_to_check = [
    "skinsakhi_mobilenetv3.keras",
    "skinsakhi_best.keras",
    "skinsakhi_experimental.keras"
]

for filename in files_to_check:
    path = os.path.join(folder, filename)

    print("\n" + "=" * 45)
    print("File:", filename)

    if not os.path.exists(path):
        print("NOT FOUND")
        continue

    print("Size:", round(os.path.getsize(path) / (1024 * 1024), 2), "MB")

    try:
        model = keras.models.load_model(path, compile=False)
        print("Input shape:", model.input_shape)
        print("Output shape:", model.output_shape)
        print("Model loads successfully")
    except Exception as e:
        print("Could not load:", str(e)[:300])



File: skinsakhi_mobilenetv3.keras
Size: 4.17 MB
Input shape: (None, 224, 224, 3)
Output shape: (None, 8)
Model loads successfully

File: skinsakhi_best.keras
Size: 4.18 MB
Input shape: (None, 224, 224, 3)
Output shape: (None, 8)
Model loads successfully

File: skinsakhi_experimental.keras
Size: 4.14 MB
Input shape: (None, 224, 224, 3)
Output shape: (None, 8)
Model loads successfully


In [ ]:

import json

notebook_path = "/content/drive/MyDrive/SkinSakhi_Final.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

for i, cell in enumerate(notebook.get("cells", [])):
    source = "".join(cell.get("source", []))
    if any(name in source for name in [
        "skinsakhi_mobilenetv3.keras",
        "skinsakhi_best.keras",
        "skinsakhi_experimental.keras"
    ]):
        print(f"\n--- CELL {i} ---\n")
        print(source[:2500])


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/SkinSakhi_Final.ipynb'

In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for name in files:
        if name.endswith(".ipynb") and "SkinSakhi" in name:
            print(os.path.join(root, name))

/content/drive/MyDrive/Colab Notebooks/SkinSakhi_MobileNetV3.ipynb
/content/drive/MyDrive/Colab Notebooks/SkinSakhi_Final.ipynb


In [ ]:

import json

notebook_path = "/content/drive/MyDrive/Colab Notebooks/SkinSakhi_Final.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

for i, cell in enumerate(notebook.get("cells", [])):
    source = "".join(cell.get("source", []))
    if any(name in source.lower() for name in [
        "skinsakhi_mobilenetv3.keras",
        "skinsakhi_best.keras",
        "skinsakhi_experimental.keras"
    ]):
        print(f"\n--- CELL {i} ---")
        print(source[:2500])



--- CELL 4 ---

import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

print("GPU:", tf.config.list_physical_devices("GPU"))

class_names = sorted([
    name for name in os.listdir(TRAIN_DIR)
    if os.path.isdir(os.path.join(TRAIN_DIR, name))
])

train_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    class_names=class_names,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    class_names=class_names,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_ds = keras.utils.image_dataset_from_directory(
    TEST_DIR,
    class_names=class_names,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AU

In [ ]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/SkinSakhi/skinsakhi_classes.txt"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import os

SAVE_DIR = "/content/drive/MyDrive/SkinSakhi"

for name in [
    "skinsakhi_mobilenetv3.keras",
    "skinsakhi_best.keras",
    "skinsakhi_experimental.keras"
]:
    path = os.path.join(SAVE_DIR, name)
    print(
        name,
        "| Exists:", os.path.exists(path),
        "| Size:", round(os.path.getsize(path) / (1024 * 1024), 2), "MB"
        if os.path.exists(path) else ""
    )


skinsakhi_mobilenetv3.keras | Exists: True | Size: 4.17 MB
skinsakhi_best.keras | Exists: True | Size: 4.18 MB
skinsakhi_experimental.keras | Exists: True | Size: 4.14 MB


In [ ]:

from tensorflow import keras
import os

SAVE_DIR = "/content/drive/MyDrive/SkinSakhi"

model_files = [
    "skinsakhi_mobilenetv3.keras",
    "skinsakhi_best.keras",
    "skinsakhi_experimental.keras"
]

results = []

for filename in model_files:
    path = os.path.join(SAVE_DIR, filename)

    print(f"\nTesting: {filename}")
    candidate = keras.models.load_model(path, compile=False)

    candidate.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    loss, accuracy = candidate.evaluate(test_ds, verbose=0)

    results.append((filename, loss, accuracy))
    print(f"Test accuracy: {accuracy * 100:.2f}%")

print("\n===== COMPARISON =====")
for filename, loss, accuracy in sorted(
    results, key=lambda x: x[2], reverse=True
):
    print(f"{filename}: {accuracy * 100:.2f}% accuracy")



Testing: skinsakhi_mobilenetv3.keras
Test accuracy: 87.98%

Testing: skinsakhi_best.keras
Test accuracy: 87.98%

Testing: skinsakhi_experimental.keras
Test accuracy: 87.98%

===== COMPARISON =====
skinsakhi_mobilenetv3.keras: 87.98% accuracy
skinsakhi_best.keras: 87.98% accuracy
skinsakhi_experimental.keras: 87.98% accuracy


In [ ]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/SkinSakhi/skinsakhi_mobilenetv3.keras"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
from google.colab import drive
drive.mount('/content/drive')

import os

folder = '/content/drive/MyDrive/SkinSakhi'

print("Folder exists:", os.path.exists(folder))

if os.path.exists(folder):
    print(os.listdir(folder))

Mounted at /content/drive
Folder exists: True
['skinsakhi_best.keras', 'skinsakhi_mobilenetv3.keras', 'skinsakhi_classes.txt', 'skinsakhi_experimental.keras']


In [6]:
import os

MODEL_PATH = "/content/drive/MyDrive/SkinSakhi/skinsakhi_mobilenetv3.keras"
LABEL_PATH = "/content/drive/MyDrive/SkinSakhi/skinsakhi_classes.txt"

print("Model found:", os.path.isfile(MODEL_PATH))
print("Labels found:", os.path.isfile(LABEL_PATH))

Model found: True
Labels found: True


In [7]:
import os
import numpy as np
import gradio as gr
from PIL import Image
from tensorflow import keras

BASE_PATH = "/content/drive/MyDrive/SkinSakhi"
MODEL_PATH = os.path.join(BASE_PATH, "skinsakhi_mobilenetv3.keras")
LABEL_PATH = os.path.join(BASE_PATH, "skinsakhi_classes.txt")

model = keras.models.load_model(MODEL_PATH)

with open(LABEL_PATH, "r", encoding="utf-8") as f:
    class_names = [line.strip() for line in f if line.strip()]

print("Model loaded successfully!")
print("Number of classes:", len(class_names))
print(class_names)

Model loaded successfully!
Number of classes: 8
['BA- cellulitis', 'BA-impetigo', 'FU-athlete-foot', 'FU-nail-fungus', 'FU-ringworm', 'PA-cutaneous-larva-migrans', 'VI-chickenpox', 'VI-shingles']
